#### 5. Dimension → `dim_date`

##### Columns:

```text
date_key (PK) (YYYYMMDD)

date
day
month
year
quarter
week
```

---

In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================
import json
import pyspark.sql.functions as f
from common_utils.generic_functions import  casting
from common_utils.logging import get_logger

# ============================================================
# 2. LOGGER
# ============================================================
logger = get_logger("dim_date.ipynb")
logger.info("Session for dim_date started")

# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",config_path
)
with open(config_path,"r") as file:
    config = json.load(file)

logger.info("Gold configuration loaded successfully")



# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================
source_config = config["source"]
target_config= config["target"]

source_name = config["source"]["source_name"]
source_table = config["source"]["source_table"]
target_table = config["target"]["target_table"]
date_format = config["transform"]["date_format"]
date_column = source_config["date_column"]


logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)
# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading order from %s",
    source_name,
    source_table
)

date_df = spark.read.table(source_table)

logger.info(
    "[%s] date table loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM CUSTOMER
# ============================================================
def transform_date(df , config):
    logger.info(
        "[%s] Starting dim_date transformation",
        source_name
    )
    # --------------------------------------------------------
    # Get minimum and maximum dates
    # --------------------------------------------------------

    date_range = (
        df
        .select(
            f.min(
                f.to_date(
                    f.from_unixtime(
                        f.col(date_column)
                    )
                )
            ).alias("min_date"),

            f.max(
                f.to_date(
                    f.from_unixtime(
                        f.col(date_column)
                    )
                )
            ).alias("max_date")
        )
        .collect()[0]
    )


    min_date = date_range["min_date"]
    max_date = date_range["max_date"]


    logger.info(
        "[%s] Minimum order date: %s",
        source_name,
        min_date
    )
    
    logger.info(
        "[%s] Maximum order date: %s",
        source_name,
        max_date
    )


    if min_date is None or max_date is None:

        raise ValueError(
            "Unable to determine date range from sales"
        )
     # --------------------------------------------------------
    # Generate date sequence
    # --------------------------------------------------------

    date_df = (
        spark
        .createDataFrame(
            [(min_date, max_date)],
            ["min_date", "max_date"]
        )
        .select(
            f.explode(
                f.sequence(
                    f.col("min_date"),
                    f.col("max_date"),
                    f.expr("interval 1 day")
                )
            ).alias("date")
        )
    )
     # --------------------------------------------------------
    # Create date attributes
    # --------------------------------------------------------

    date_df = (
        date_df

        .withColumn(
            "date_key",
            f.date_format(
                f.col("date"),
                config["transform"]["date_format"]
            ).cast("int")
        )

        .withColumn(
            "day",
            f.dayofmonth(
                f.col("date")
            )
        )

        .withColumn(
            "month",
            f.month(
                f.col("date")
            )
        )
        .withColumn(
            "year",
            f.year(
                f.col("date")
            )
        )

        .withColumn(
            "quarter",
            f.quarter(
                f.col("date")
            )
        )

        .withColumn(
            "week",
            f.weekofyear(
                f.col("date")
            )
        )
    )

     # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    date_df = casting(
        date_df,
        config.get("cast_columns", {})
    )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    date_df = date_df.select(
        *config["select"]
    )


    logger.info(
        "[%s] dim_date transformation completed",
        source_name
    )

    return date_df

    
# ============================================================
# 7. CREATE DIM ORDER
# ============================================================

logger.info(
    "[%s] Starting dim_order creation",
    source_name
)

dim_date = transform_date(date_df,config)




# ============================================================
# 8. VALIDATE DATA
# ============================================================
logger.info(
    "[%s] Validating dim_date",
    source_name
)

records_count=dim_date.count()
if records_count==0:
    raise ValueError("no record found in Dim_date")

duplicate_count = (
    dim_date
    .groupBy("date_key")
    .count()
    .filter(f.col("count") > 1)
    .count()
)
if duplicate_count > 0:
    raise ValueError ("Duplicae detected")


# ============================================================
# 9. DISPLAY
# ============================================================
dim_date.display()




# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================
logger.info(
    "[%s] Creating Gold schema if required",
    source_name
)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS 
    retail_data_platform.gold
    
          """)
# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_date to %s",
    source_name,
    target_table
)

(dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table ))

logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)

# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_date record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_order pipeline completed successfully",
    source_name
)








 
    
    

